# 📊 Data Analysis — 10-Week Course
## Week 9: Advanced Topics

---
### Learning Objectives
By the end of this week, you will be able to:
- Apply advanced feature engineering techniques
- Use dimensionality reduction (PCA, t-SNE) for high-dimensional data
- Query and integrate data with SQL and Pandas
- Interpret real-world case studies applying the full analysis pipeline

### Outline
1. Advanced Feature Engineering
2. Dimensionality Reduction
3. SQL & Pandas Integration
4. Cross-Validation & Model Selection
5. Case Study: Building a Country Health Index
6. Lab Exercises
7. Assignment

---
## 1. Advanced Feature Engineering

Feature engineering transforms raw data into representations that machine learning models can better use.

### Techniques
| Technique | Example | Purpose |
|---|---|---|
| **Log transform** | `log(GDP)` | Normalise right-skewed variables |
| **Interaction terms** | `GDP × Vaccination` | Capture joint effects |
| **Polynomial features** | `water_access²` | Non-linear relationships |
| **Binning** | GDP terciles | Convert continuous → ordinal |
| **Composite index** | (LE + Vax + Water) / 3 | Summarise related indicators |
| **Lag / rolling mean** | 3-year average | Smooth time series |
| **Rank** | Country rank by LE | Relative position |
| **One-hot encoding** | Region dummies | Represent categories numerically |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, PolynomialFeatures
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import (
    cross_val_score, GridSearchCV, KFold
)
from sklearn.metrics import r2_score, mean_squared_error
import sqlite3, warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

import os
if os.path.exists("africa_health_cleaned.csv"):
    df = pd.read_csv("africa_health_cleaned.csv")
else:
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    })

print("Dataset ready:", df.shape)

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────
df2 = df.copy()

# 1. Log transforms
df2["log_gdp"]          = np.log(df2["gdp_per_capita"])
df2["log_infant_mort"]  = np.log(df2["infant_mortality"])
df2["log_maternal_mort"]= np.log(df2["maternal_mortality"])

# 2. Interaction term: health spend × log GDP (resource efficiency)
df2["spend_gdp_interact"] = df2["health_expenditure"] * df2["log_gdp"]

# 3. Composite Health Index (0–100)
#    Higher LE → higher; higher infant mortality → lower; higher vax → higher
scaler_mm = MinMaxScaler()
df2["le_norm"]   = scaler_mm.fit_transform(df2[["life_expectancy"]])
df2["im_norm"]   = 1 - scaler_mm.fit_transform(df2[["infant_mortality"]])  # invert
df2["vax_norm"]  = scaler_mm.fit_transform(df2[["vaccination_dtp3"]])
df2["wat_norm"]  = scaler_mm.fit_transform(df2[["water_access"]])
df2["health_index"] = (
    0.35 * df2["le_norm"] +
    0.25 * df2["im_norm"] +
    0.20 * df2["vax_norm"] +
    0.20 * df2["wat_norm"]
) * 100

# 4. Region rank per LE (within region)
df2["le_region_rank"] = df2.groupby("region")["life_expectancy"].rank(ascending=False)

# 5. GDP quartile
df2["gdp_quartile"] = pd.qcut(df2["gdp_per_capita"], q=4,
                                labels=["Q1 (Low)","Q2","Q3","Q4 (High)"])

# 6. One-hot encode region
region_dummies = pd.get_dummies(df2["region"], prefix="region", drop_first=True)
df2 = pd.concat([df2, region_dummies], axis=1)

print("New features added:")
new_feats = ["log_gdp","log_infant_mort","log_maternal_mort",
             "spend_gdp_interact","health_index","le_region_rank","gdp_quartile"]
print(df2[new_feats].describe().round(2))

print("\nTop 10 countries by Health Index:")
print(df2[["country","region","health_index","life_expectancy"]]
      .nlargest(10, "health_index").to_string(index=False))

---
## 2. Dimensionality Reduction

High-dimensional datasets are hard to visualise and can suffer from the **curse of dimensionality**.

### PCA (Principal Component Analysis)
- Finds orthogonal axes of maximum variance
- Preserves **global structure**
- Interpretable components

### t-SNE (t-distributed Stochastic Neighbor Embedding)
- Preserves **local structure** (neighbourhood relationships)
- Excellent for visualisation
- Non-deterministic; not for general dimensionality reduction

In [ ]:
PALETTE = {
    "West Africa": "#3498DB", "East Africa": "#2ECC71",
    "North Africa": "#E74C3C", "Central Africa": "#9B59B6",
    "Southern Africa": "#F39C12",
}

pca_features = ["log_gdp", "life_expectancy", "log_infant_mort",
                "vaccination_dtp3", "water_access",
                "health_expenditure", "hiv_prevalence"]

X_pca_input = df2[pca_features].dropna()
idx = X_pca_input.index
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca_input)

# Full PCA
pca_full = PCA()
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

# 2-component PCA
pca2 = PCA(n_components=2)
X_pc  = pca2.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# A — Scree plot
ax = axes[0]
ax.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
       pca_full.explained_variance_ratio_ * 100,
       color="#3498DB", alpha=0.8, edgecolor="white")
ax.plot(range(1, len(cumvar)+1), cumvar * 100,
        marker="o", color="#E74C3C", lw=2, label="Cumulative")
ax.axhline(80, color="gray", ls="--", lw=1, label="80% threshold")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance (%)")
ax.set_title("A — Scree Plot", fontweight="bold")
ax.legend()

# B — Biplot (PC1 vs PC2)
ax = axes[1]
for region in df2.loc[idx, "region"].unique():
    mask = df2.loc[idx, "region"] == region
    ax.scatter(X_pc[mask, 0], X_pc[mask, 1],
               color=PALETTE[region], label=region,
               alpha=0.8, edgecolors="white", s=70)

# Loading vectors
scale = 2.5
for feat, vec in zip(pca_features, pca2.components_.T):
    ax.arrow(0, 0, vec[0]*scale, vec[1]*scale,
             head_width=0.1, head_length=0.05,
             fc="black", ec="black", alpha=0.5)
    ax.text(vec[0]*scale*1.1, vec[1]*scale*1.1, feat,
            fontsize=7, ha="center", color="#2C3E50")

ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("B — PCA Biplot", fontweight="bold")
ax.legend(fontsize=7)

# C — t-SNE
ax = axes[2]
tsne = TSNE(n_components=2, perplexity=15, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)
for region in df2.loc[idx, "region"].unique():
    mask = df2.loc[idx, "region"] == region
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               color=PALETTE[region], label=region,
               alpha=0.8, edgecolors="white", s=70)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("C — t-SNE Projection", fontweight="bold")
ax.legend(fontsize=7)

plt.suptitle("Dimensionality Reduction — Africa Health",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week9_dimred.png", bbox_inches="tight")
plt.show()

---
## 3. SQL & Pandas Integration

Many real-world datasets live in databases. Python bridges Pandas and SQL seamlessly.

In [ ]:
# Build an in-memory SQLite database from our dataset
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

# Write DataFrame to SQL
df2[["country","region","life_expectancy","infant_mortality",
     "gdp_per_capita","vaccination_dtp3","water_access",
     "health_expenditure","hiv_prevalence","health_index"]].to_sql(
    "health", conn, index=False, if_exists="replace"
)

# Also write a separate regions lookup table
region_info = pd.DataFrame({
    "region"   : list(PALETTE.keys()),
    "continent": ["Africa"] * 5,
    "num_countries": [16, 14, 6, 8, 10],
})
region_info.to_sql("regions", conn, index=False, if_exists="replace")

print("Database tables:")
for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'"):
    print(" •", row["name"])

In [ ]:
# SQL queries with pd.read_sql

# Query 1: Top 5 countries by health index
q1 = """SELECT country, region, ROUND(health_index, 2) AS health_index,
               ROUND(life_expectancy, 1) AS life_expectancy
         FROM health
         ORDER BY health_index DESC
         LIMIT 5"""
print("Top 5 Countries by Health Index:")
print(pd.read_sql(q1, conn).to_string(index=False))

# Query 2: Regional aggregation
q2 = """SELECT region,
               COUNT(*) AS n_countries,
               ROUND(AVG(life_expectancy), 1) AS avg_le,
               ROUND(AVG(health_index), 1) AS avg_index,
               ROUND(MIN(gdp_per_capita), 0) AS min_gdp,
               ROUND(MAX(gdp_per_capita), 0) AS max_gdp
         FROM health
         GROUP BY region
         ORDER BY avg_le DESC"""
print("\nRegional Summary:")
print(pd.read_sql(q2, conn).to_string(index=False))

# Query 3: JOIN health with regions table
q3 = """SELECT h.country, h.region, r.num_countries,
               ROUND(h.health_index, 1) AS health_index,
               ROUND(h.gdp_per_capita, 0) AS gdp
         FROM health h
         JOIN regions r ON h.region = r.region
         WHERE h.gdp_per_capita > 3000
         ORDER BY h.health_index DESC"""
print("\nHigh-GDP Countries (JOIN with regions):")
print(pd.read_sql(q3, conn).to_string(index=False))

---
## 4. Cross-Validation & Model Selection

### Why Cross-Validation?
A single train/test split can give misleading estimates — especially with small datasets.  
**k-fold CV** trains k times, each time using a different fold as the test set.

### Regularisation
| Method | Penalty | Effect |
|---|---|---|
| **Ridge (L2)** | λ·Σβ² | Shrinks all coefficients toward zero |
| **Lasso (L1)** | λ·Σ|β| | Shrinks some coefficients to exactly zero (feature selection) |

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

feat_cols = ["log_gdp", "vaccination_dtp3", "water_access",
             "health_expenditure", "log_infant_mort",
             "spend_gdp_interact"]

df2["log_gdp"]            = np.log(df2["gdp_per_capita"])
df2["log_infant_mort"]    = np.log(df2["infant_mortality"])
df2["spend_gdp_interact"] = df2["health_expenditure"] * df2["log_gdp"]

X_reg = df2[feat_cols].dropna()
y_reg = df2.loc[X_reg.index, "life_expectancy"]

scaler_r = StandardScaler()
X_reg_s = scaler_r.fit_transform(X_reg)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "OLS"          : LinearRegression(),
    "Ridge α=0.1"  : Ridge(alpha=0.1),
    "Ridge α=10"   : Ridge(alpha=10),
    "Lasso α=0.1"  : Lasso(alpha=0.1, max_iter=5000),
    "Lasso α=1.0"  : Lasso(alpha=1.0, max_iter=5000),
}

print(f"{'Model':<18} {'CV R²':>8} {'CV RMSE':>10} {'Coefficients > 0':>16}")
print("-" * 60)

for name, model in models.items():
    cv_r2   = cross_val_score(model, X_reg_s, y_reg, cv=kf, scoring="r2").mean()
    cv_neg_mse = cross_val_score(model, X_reg_s, y_reg, cv=kf,
                                  scoring="neg_mean_squared_error").mean()
    cv_rmse = np.sqrt(-cv_neg_mse)

    model.fit(X_reg_s, y_reg)
    coefs = model.coef_
    n_nonzero = np.sum(np.abs(coefs) > 0.001)

    print(f"{name:<18} {cv_r2:>8.4f} {cv_rmse:>10.4f} {n_nonzero:>16}")

print(f"\nFeature names: {feat_cols}")

In [ ]:
# Lasso regularisation path — coefficient vs alpha
alphas = np.logspace(-2, 1, 60)
coef_matrix = []
for a in alphas:
    lasso = Lasso(alpha=a, max_iter=5000)
    lasso.fit(X_reg_s, y_reg)
    coef_matrix.append(lasso.coef_.copy())
coef_matrix = np.array(coef_matrix)

fig, ax = plt.subplots(figsize=(10, 5))
for i, feat in enumerate(feat_cols):
    ax.plot(np.log10(alphas), coef_matrix[:, i], lw=2, label=feat)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_xlabel("log₁₀(α)")
ax.set_ylabel("Coefficient Value")
ax.set_title("Lasso Regularisation Path — Feature Selection",
             fontweight="bold")
ax.legend(fontsize=8, loc="right")
plt.tight_layout()
plt.savefig("week9_lasso_path.png", bbox_inches="tight")
plt.show()
print("As α increases, Lasso shrinks coefficients to zero → automatic feature selection")

---
## 5. Case Study: Building a Country Health Index

This case study applies the **full 7-step analysis workflow** to a real question:

> **"Which African countries have the best overall health outcomes, and what drives them?"

Steps: Define → Collect → Clean → Explore → Analyse → Visualise → Communicate

In [ ]:
print("CASE STUDY: Africa Country Health Index")
print("=" * 55)

# ── Step 1: Define ────────────────────────────────────────────────────
print("\n[Step 1] Research Question")
print("  Which African countries have the best health outcomes,")
print("  and can we predict the health index from economic")
print("  and infrastructure variables?")

# ── Step 2: Dataset overview ──────────────────────────────────────────
print(f"\n[Step 2] Dataset: {df2.shape[0]} countries × {df2.shape[1]} features")
print(f"  Missing values : {df2.isnull().sum().sum()}")

# ── Step 3: EDA summary ───────────────────────────────────────────────
print("\n[Step 3] Descriptive Stats — Health Index")
hi = df2["health_index"]
print(f"  Mean   = {hi.mean():.1f}")
print(f"  Median = {hi.median():.1f}")
print(f"  Std    = {hi.std():.1f}")
print(f"  Range  = [{hi.min():.1f}, {hi.max():.1f}]")

# ── Step 4: Statistical analysis ──────────────────────────────────────
print("\n[Step 4] Correlation: health_index vs log_gdp")
r, p = stats.pearsonr(df2["health_index"], df2["log_gdp"])
print(f"  Pearson r = {r:.4f},  p = {p:.6f}")
print(f"  → {'Strong positive association' if r > 0.5 else 'Moderate association'}")

# ── Step 5: Model — predict health index ─────────────────────────────
print("\n[Step 5] MLR Model — Predict Health Index")
from sklearn.linear_model import LinearRegression
model_features = ["log_gdp", "vaccination_dtp3", "water_access", "health_expenditure"]
X_cs = df2[model_features].dropna()
y_cs = df2.loc[X_cs.index, "health_index"]
model_cs = LinearRegression()
cv_r2 = cross_val_score(model_cs, X_cs, y_cs, cv=5, scoring="r2").mean()
print(f"  5-fold CV R² = {cv_r2:.4f}")
model_cs.fit(X_cs, y_cs)
for feat, coef in zip(model_features, model_cs.coef_):
    print(f"  {feat:<25} β = {coef:+.3f}")

# ── Step 6: Top & bottom countries ────────────────────────────────────
print("\n[Step 6] Top 5 and Bottom 5 by Health Index")
ranked = df2[["country","region","health_index","life_expectancy","gdp_per_capita"]]
ranked = ranked.sort_values("health_index", ascending=False)
print("  TOP 5:")
print(ranked.head(5).to_string(index=False))
print("  BOTTOM 5:")
print(ranked.tail(5).to_string(index=False))

print("\n[Step 7] Key Insights")
print("  1. Log GDP is the strongest predictor of the health index")
print("  2. North African countries dominate the top ranks")
print("  3. Water access explains variance not captured by GDP alone")
print("  4. CV R² > 0.70 suggests the model is practically useful")

In [ ]:
# Case study visualisation: 3-panel dashboard
fig = plt.figure(figsize=(17, 6))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# A — Health Index ranking
ax_a = fig.add_subplot(gs[0, 0])
top10 = ranked.head(10)
colors_top = [PALETTE.get(r, "#95A5A6") for r in top10["region"]]
ax_a.barh(top10["country"][::-1], top10["health_index"][::-1],
          color=colors_top[::-1], edgecolor="white", alpha=0.9)
ax_a.set_xlabel("Health Index (0–100)")
ax_a.set_title("A — Top 10 by Health Index", fontweight="bold")

# B — Health index vs log GDP scatter
ax_b = fig.add_subplot(gs[0, 1])
for region, grp in df2.groupby("region"):
    ax_b.scatter(grp["log_gdp"], grp["health_index"],
                 color=PALETTE[region], alpha=0.8, s=60,
                 edgecolors="white", label=region)
# Fit line
m, b = np.polyfit(df2["log_gdp"], df2["health_index"], 1)
x_fit = np.linspace(df2["log_gdp"].min(), df2["log_gdp"].max(), 100)
ax_b.plot(x_fit, m*x_fit+b, "k--", lw=1.5)
ax_b.set_xlabel("Log GDP per Capita")
ax_b.set_ylabel("Health Index")
ax_b.set_title(f"B — Health Index vs GDP (r={r:.2f})", fontweight="bold")
ax_b.legend(fontsize=6)

# C — Regional box
ax_c = fig.add_subplot(gs[0, 2])
region_order = df2.groupby("region")["health_index"].median().sort_values().index
sns.boxplot(data=df2, x="region", y="health_index",
            order=region_order, palette=PALETTE, ax=ax_c)
ax_c.set_title("C — Regional Distribution", fontweight="bold")
ax_c.tick_params(axis="x", rotation=20)
ax_c.set_xlabel("")

fig.suptitle("Case Study: Africa Country Health Index",
             fontsize=14, fontweight="bold")
plt.savefig("week9_case_study.png", bbox_inches="tight")
plt.show()

---
## 6. Lab Exercises

### Lab 1: Feature Engineering
Create three new features and add them to `df2`:
- `infant_per_gdp`: `infant_mortality / gdp_per_capita * 1000`
- `health_eff`: `life_expectancy / health_expenditure` (years per $1% GDP)
- `water_vax_score`: geometric mean of `water_access` and `vaccination_dtp3`

In [ ]:
# Lab 1
# TODO: Add 3 new engineered features, print describe()

### Lab 2: PCA Loadings Interpretation
From the PCA biplot, identify:
- Which features contribute most to PC1?
- What does PC1 represent conceptually?
- How much cumulative variance do 2 components explain?

In [ ]:
# Lab 2 — Print PCA loadings table
loadings = pd.DataFrame(
    pca2.components_.T,
    index=pca_features,
    columns=["PC1", "PC2"]
).round(3)
print("PCA Loadings:")
print(loadings)
# TODO: Identify dominant features and write 2-sentence interpretation

### Lab 3: GridSearchCV for Ridge
Use `GridSearchCV` with 5-fold CV to find the best `alpha` for Ridge regression.  
Search over: `[0.001, 0.01, 0.1, 1, 10, 100]`

In [ ]:
# Lab 3
param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
# TODO: GridSearchCV → Ridge → best_params_, best_score_

---
## 7. Assignment — Week 9

**Problem 1 (25 pts):** Build a composite **Disease Burden Index**:  
`burden = 0.4 × log_infant_mort_norm + 0.35 × hiv_norm + 0.25 × maternal_mort_norm`  
(each normalised 0–1 with higher = worse).  
Rank all 54 countries. Visualise in a diverging bar chart (centred on median).

**Problem 2 (25 pts):** Run PCA on all 7 numeric health variables.  
Create a scree plot showing how many components are needed for ≥ 85% variance.  
Interpret what PC1 and PC2 represent based on loadings.

**Problem 3 (25 pts):** Write 5 SQL queries on the SQLite database:  
1. Countries where both `life_expectancy > 65` AND `vaccination_dtp3 > 85`  
2. Average `health_index` per region, ordered descending  
3. Countries with `hiv_prevalence > 10` (worst burden)  
4. Country with highest and lowest GDP per region  
5. A query of your choice with a JOIN

**Problem 4 (25 pts):** Implement a `full_pipeline(df, target, features)` function that:  
- Scales features  
- Runs OLS, Ridge, Lasso with GridSearchCV  
- Returns a comparison DataFrame  
- Plots the Lasso regularisation path

In [ ]:
# Problem 1
pass  # TODO

In [ ]:
# Problem 2
pass  # TODO

In [ ]:
# Problem 3 — 5 SQL queries
queries = [
    # 1
    "SELECT country FROM health WHERE life_expectancy > 65 AND vaccination_dtp3 > 85",
    # 2 — TODO
    # 3 — TODO
    # 4 — TODO
    # 5 — TODO
]
for i, q in enumerate(queries, 1):
    print(f"Query {i}:")
    print(pd.read_sql(q, conn).to_string(index=False))
    print()

In [ ]:
# Problem 4
def full_pipeline(df, target, features):
    # TODO
    pass

---
## Summary — Week 9

| Concept | Key Point |
|---|---|
| Feature engineering | Log transforms, interaction terms, composite indices, one-hot encoding |
| PCA | Unsupervised; maximises explained variance; biplot shows loadings |
| t-SNE | Preserves local structure; best for visualisation only |
| SQL + Pandas | `pd.read_sql()` bridges databases and DataFrames |
| Cross-validation | k-fold CV gives unbiased performance estimate |
| Ridge (L2) | Shrinks all coefficients; handles multicollinearity |
| Lasso (L1) | Can zero out coefficients — automatic feature selection |
| GridSearchCV | Systematic hyperparameter tuning with cross-validation |

**Next:** Week 10 — Capstone Project